In [ ]:
import pandas as pd

df = pd.read_json("/workspace/daeyong/fourth_finetuning_data/2wiki_added_ver3.json")
df

In [ ]:
df['current_step'].apply(lambda x: x.replace("Step ", "")[0]).value_counts()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb

# 1) 데이터 정의 (기존 유지)
data_structure = {
    'Correct': {'Correct': 3930},
    'Procedural': {
        'Overthinking': 923, 'Inefficiency': 837, 
        'Off-topic': 681, 'Redundancy': 648
    },
    'Attribution': {
        'Unsupported': 904, 'Premature\nAttribution': 787, 
        'Information\nMiss': 659, 'Contradictory': 612
    },
    'Logical': {'Logical Fallacy': 1120},
    'Final\nAnswer': {'Wrong\nConclusion': 414}
}

# helper 함수들 유지
def blend_with_white(rgb, t):
    r, g, b = rgb
    return (r + (1 - r) * t, g + (1 - g) * t, b + (1 - b) * t)

def make_sub_shades(main_color, n_sub):
    base = to_rgb(main_color)
    if n_sub == 1: return [blend_with_white(base, 0.32)]
    ts = np.linspace(0.18, 0.60, n_sub)
    return [blend_with_white(base, float(t)) for t in ts]

# 2) 데이터 가공
inner_labels, inner_counts = [], []
outer_labels, outer_counts, outer_colors = [], [], []
main_colors = ["#0072B2", "#009E73", "#D55E00", "#CC79A7", "#F0E442"]

for i, (main_cat, sub_dict) in enumerate(data_structure.items()):
    sub_items = list(sub_dict.items())
    inner_labels.append(main_cat)
    inner_counts.append(sum(v for _, v in sub_items))
    sub_shades = make_sub_shades(main_colors[i], len(sub_items))
    for (sub_cat, count), col in zip(sub_items, sub_shades):
        outer_labels.append(sub_cat)
        outer_counts.append(count)
        outer_colors.append(col)

total_n = sum(inner_counts)

# 3) 차트 생성 - "줌인" 및 "여백 제거" 핵심 로직
fig, ax = plt.subplots(figsize=(8, 8))

# ✅ 여백 제거의 정석
# tight_layout을 미리 적용하고, savefig에서 pad_inches를 0으로 주는 것이 가장 확실합니다.
plt.subplots_adjust(left=0, right=1, bottom=0, top=1) 

START_ANGLE = 140
outer_width = 0.30
inner_width = 0.30
EDGE = "#F2F2F2"

# 바깥 원: radius를 1로 설정
out_pie, _ = ax.pie(
    outer_counts, radius=1.0, colors=outer_colors, startangle=START_ANGLE,
    counterclock=True, wedgeprops=dict(width=outer_width, edgecolor=EDGE, linewidth=1.2)
)

# 안쪽 원: radius를 0.7로 설정
in_pie, _ = ax.pie(
    inner_counts, radius=0.70, colors=main_colors, startangle=START_ANGLE,
    counterclock=True, wedgeprops=dict(width=inner_width, edgecolor=EDGE, linewidth=1.2)
)

# 4) 라벨 배치 (기존 위치 유지)
r_label_outer = 1.0 - outer_width / 2.0
r_label_inner = 0.70 - inner_width / 2.0

for i, p in enumerate(out_pie):
    ang = np.deg2rad((p.theta2 + p.theta1) / 2.0)
    x, y = r_label_outer * np.cos(ang), r_label_outer * np.sin(ang)
    pct = (outer_counts[i] / total_n) * 100
    span = abs(p.theta2 - p.theta1)
    fs = 8 if span < 10 else (9 if span < 16 else 10)
    ax.text(x, y, f"{outer_labels[i]}\n{pct:.1f}%", ha='center', va='center', fontsize=fs, fontweight='medium')

for i, p in enumerate(in_pie):
    ang = np.deg2rad((p.theta2 + p.theta1) / 2.0)
    x, y = r_label_inner * np.cos(ang), r_label_inner * np.sin(ang)
    pct = (inner_counts[i] / total_n) * 100
    ax.text(x, y, f"{inner_labels[i]}\n{pct:.1f}%", ha='center', va='center', fontsize=11, fontweight='medium')

# 중앙 제목
ax.text(0, 0, "Error Statistics\nin Training Set\n\nN=11,515", 
        ha='center', va='center', fontsize=14, fontweight='bold', linespacing=1.2)

# ✅ 차트가 캔버스 끝까지 닿도록 좌표축 범위 강제 설정 (-1에서 1까지)
ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.axis('off')

# 5) 저장 시 여백 완전 박멸
# 저장할 때 pad_inches=0을 주면 흰색 테두리가 사라집니다.
plt.savefig('training_data_statistics.png', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 1. 데이터 정의 및 그룹화 (6단계 이상 통합)
raw_step_data = {
    1: 2285, 2: 2037, 3: 2090, 4: 3104,
    5: 1627, 6: 364, 7: 7, 8: 1
}

# 1~5단계는 유지, 6단계 이상(6, 7, 8)은 합산
processed_data = {}
step_6_plus_sum = 0

for step, count in raw_step_data.items():
    if step < 6:
        processed_data[f"Step {step}"] = count
    else:
        step_6_plus_sum += count

processed_data["Step 6+"] = step_6_plus_sum

# 2. 리스트로 변환
labels = list(processed_data.keys())
counts = list(processed_data.values())

# 3. 시각화 설정 (학술지 스타일)
plt.figure(figsize=(10, 6))
color = "#6DB748" # 학술용 파란색 유지

bars = plt.bar(labels, counts, color=color, alpha=0.8, edgecolor='black', linewidth=1)

# 막대 상단에 구체적인 수치 표시
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 20,
             f'{int(height):,}', ha='center', va='bottom', fontsize=12, fontweight='medium')

# 레이블 및 격자 설정
plt.xlabel('Step Position', fontsize=14, fontweight='bold', labelpad=10)
plt.ylabel('Number of Datas', fontsize=14, fontweight='bold', labelpad=10)
plt.title('Distribution of Error Step Positions', 
          fontsize=14, fontweight='bold', pad=20)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.ylim(0, max(counts) * 1.15) # 상단 수치 표시를 위한 여백 확보

# 테두리 정리
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()

# 4. 저장 (고해상도 300dpi)
plt.savefig('error_position_statistics.png', dpi=300, bbox_inches='tight')
plt.show()